This notebook decomposes `HeatingSystem` into its component parts using the same four structural constructs introduced in Chapter 1; after running it you can see the same abstract-def, part-def, specialization, and composition pattern applied one level down.

Chapter 1 built the toaster's top-level structure: an abstract base, concrete types, specialization, and composition. Chapter 6 applies those four constructs to decompose `HeatingSystem` into a `ResistanceCoil` and a `PowerWire`, both specializations of `HeatingElement`.

A `HeatingAssembly` part definition composes them — it specializes `HeatingSystem` and owns both subparts. `model.query()` can then return the full set of part definitions at this level.

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch06-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch06-cumulative.sysml` file adds a second-level requirement: `HeatingReq` constrains `heater.power >= 600.0` W on the `Heater` sub-component. A `HeatingAssembly` decomposition adds `ResistanceCoil` and `PowerWire` sub-parts. Two candidate heaters — `efficient` (800 W) and `weak` (400 W) — exercise the new requirement using the same satisfy-assertion pattern from Chapter 3, applied one level down in the hierarchy.

In [ ]:
# Negative control: composing a part typed by an undefined type raises "unresolved reference".
# Composition requires the type to be declared — the same rule applies at every level.
bad_source = """
package BadSecond {
    private import ScalarValues::*;
    abstract part def HeatingElement;
    part def ResistanceCoil :> HeatingElement;
    part def HeatingAssembly {
        part coil : ResistanceCoil;
        part wire : UndefinedType;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print(f"Neg control diagnostics: {bad.diagnostics[0].message!r}")

In [ ]:
# List all PartDefinition elements — should include the second-level types
part_defs = [e.as_dict() for e in model.query()
             if e.as_dict().get("@type") == "PartDefinition"]
for pd in part_defs:
    name = pd.get("declaredName") or pd.get("name", "?")
    abstract = pd.get("isAbstract") == "true"
    print(f"  {'abstract ' if abstract else ''}part def {name}")

The four structural constructs from Chapter 1 (A-F) are applied one level down in the model hierarchy and parsed by OpenSysML (O-S); `model.query()` returns all `PartDefinition` elements including the second-level `HeatingElement`, `ResistanceCoil`, `PowerWire`, and `HeatingAssembly` (E).

Try the chapter exercise in `exercises/ch06/exercise.ipynb`: decompose `BrewUnit` into an `Impeller` and a `FilterBasket`, both specializations of a `BrewComponent` abstract part, and confirm all three appear in the `model.query()` result.